# Time-evolved expectation values with `EulerIntegrator`

This notebook demonstrates how to:

1. Define a driven qubit Hamiltonian $H(t) = H_0 + f(t)\,H_1$
2. Build a Trotter evolution circuit via `EulerIntegrator`
3. Run **noiseless** and **noisy** circuit simulations using the QDK backend
4. Compare against a fine-grained matrix-exponential reference

In [ ]:
import numpy as np

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef, DrivenQubitHamiltonian, LatticeGraph, QuantumErrorProfile,
    QubitHamiltonian,
)
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)


## Define the driven Hamiltonian

$$H(t) = H_0 + f(t)\,H_1$$

| Component | Expression | Role |
|-----------|-----------|------|
| $H_0$ | $J\,Z_0Z_1$ | Static Ising ZZ coupling |
| $H_1$ | $h\,(X_0+X_1)$ | Transverse field |
| $f(t)$ | $\sin(\omega\,t)$ | Smooth sinusoidal drive |

In [ ]:
lattice = LatticeGraph.chain(2)
h0 = create_ising_hamiltonian(lattice, j=1.0, h=0.0)   # ZZ coupling
h1 = create_ising_hamiltonian(lattice, j=0.0, h=0.5)   # transverse field

omega = 2 * np.pi
total_time = 2.0
dt = 0.1 # Time step for evolution discretization
num_divisions = 2 # Trotter subdivisions within each dt step

td_hamiltonian = DrivenQubitHamiltonian(h0, h1, drive=lambda t: np.sin(omega * t))
observable = QubitHamiltonian(["ZZ"], np.array([1.0]))

In [ ]:
from qdk_chemistry.algorithms.state_preparation import identity_state_prep

# Identity state prep: no gates before evolution → initial state is |0...0⟩ (all spins up in Z basis)
state_prep = identity_state_prep(num_qubits=td_hamiltonian.num_qubits)

## Reference evolution (numerical exponentiation)

High-accuracy reference via fine-grained Magnus order-1 propagation: each `dt` interval uses the time-averaged Hamiltonian $H_0 + \tilde f\,H_1$ where $\tilde f = \frac{1}{\Delta t}\int f(t')\,dt'$, then subdivided into fine substeps.

In [ ]:
from scipy.integrate import quad
from scipy.sparse.linalg import expm_multiply

H0_sp, H1_sp = h0.to_matrix(sparse=True), h1.to_matrix(sparse=True)
drive = td_hamiltonian.drive

psi = np.zeros(2**h0.num_qubits, dtype=complex)
psi[0] = 1.0

n_steps = max(1, round(total_time / dt))
n_substeps = 100

for step in range(n_steps):
    t_start = step * dt
    t_end = t_start + dt
    f_avg, _ = quad(drive, t_start, t_end)
    f_avg /= dt
    H_eff = H0_sp + f_avg * H1_sp
    dt_fine = dt / n_substeps
    for _ in range(n_substeps):
        psi = expm_multiply(-1j * H_eff * dt_fine, psi)

ZZ_diag = np.kron([1, -1], [1, -1]).astype(complex)
zz_ref = np.real(psi.conj() @ (ZZ_diag * psi))
print(f"Reference ⟨ZZ⟩: {zz_ref:.6f}")

## Noiseless Trotter simulation

Configure `EulerIntegrator` with a Magnus order-1 propagator: the algorithm divides `[0, total_time]` into steps of size `dt`, computes the time-averaged $H_\text{eff}$ per interval, and Trotterizes each step with `num_divisions` subdivisions.

In [ ]:
shots = 100000

hamiltonian_simulation = create("hamiltonian_simulation", "euler_integrator")
hamiltonian_simulation.settings().set("evolution_builder", AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=num_divisions, order=1))
hamiltonian_simulation.settings().set("circuit_executor", AlgorithmRef("circuit_executor", "qdk_full_state_simulator"))
hamiltonian_simulation.settings().set("propagator", AlgorithmRef("propagator", "magnus", order=1))
hamiltonian_simulation.settings().set("total_time", total_time)
hamiltonian_simulation.settings().set("dt", dt)

In [ ]:
results_noiseless = hamiltonian_simulation.run(td_hamiltonian, observables=[observable], state_prep=state_prep, shots=shots)
zz_noiseless = results_noiseless[0][0].energy_expectation_value
print(f"Noiseless ⟨ZZ⟩: {zz_noiseless:.6f}")

## Noisy simulations

Define a light depolarizing noise profile and run on the QDK backends.

In [ ]:
qdk_error_profile = QuantumErrorProfile(
    name="demo_noise",
    description="Light depolarizing noise on common gates",
    errors={
        "cx":  {"depolarizing_error": 0.001},
        "cz":  {"depolarizing_error": 0.001},
        "rzz": {"depolarizing_error": 0.001},
        "h":   {"depolarizing_error": 0.001},
        "rz":  {"depolarizing_error": 0.001},
        "rx":  {"depolarizing_error": 0.001},
        "s":   {"depolarizing_error": 0.001},
        "sdg": {"depolarizing_error": 0.001},
    },
)

In [ ]:
results_noisy_qdk = hamiltonian_simulation.run(td_hamiltonian, observables=[observable], state_prep=state_prep, shots=shots, noise=qdk_error_profile)
zz_noisy_qdk = results_noisy_qdk[0][0].energy_expectation_value
print(f"Noisy QDK ⟨ZZ⟩: {zz_noisy_qdk:.6f}")

## Results

In [ ]:
print("Simulator               ⟨ZZ⟩")
print("─" * 36)
print(f"Reference                 {zz_ref: .6f}")
print(f"QDK  (noiseless)      {zz_noiseless: .6f}")
print(f"QDK  (noisy)          {zz_noisy_qdk: .6f}")